In [ ]:
# @title Start FreeFakeStudio
# @markdown ---
# @markdown ### Persistent Colab settings
WORKSPACE_DIR = "/content/drive/MyDrive/FreeFakeStudio"  # @param {type:"string"}
# @markdown Drive folder for app source, ComfyUI, models, cache, and results.
UPDATE_APP = False  # @param {type:"boolean"}
# @markdown Optional: fast-forward update the Drive app copy. No hard reset.
REPAIR_INSTALL = False  # @param {type:"boolean"}
# @markdown Optional: re-check/re-download missing or suspicious files.
NGROK_AUTH_TOKEN = ""  # @param {type:"string"}
# @markdown Recommended: use ngrok for a clean public HTTPS URL. Leave blank to use Colab's signed-in HTTPS proxy.
# @markdown ---
# @markdown ### FLUX.2 Klein text encoder
FLUX_ENCODER = "Official"  # @param ["Official", "Custom"]
# @markdown Official is the memory-tested FP4 encoder. Custom is used only after a compatible file URL is supplied below.
FLUX_CUSTOM_ENCODER_URL = "https://huggingface.co/ponpoke/flux2-klein-4b-uncensored-text-encoder/resolve/main/flux2-klein-4b-uncensored-q4_k_m.gguf"  # @param {type:"string"}
# @markdown Hugging Face file URL ending in .gguf or .safetensors. Recommended: Q4, 2.0-2.7 GiB. Hard limit: 3.0 GiB.
HUGGINGFACE_TOKEN = ""  # @param {type:"string"}
# @markdown Optional token for a gated repository whose terms you have accepted. It is not written to diagnostics.
# @markdown ---
# @markdown ### Avatar Studio APIs
GEMINI_API_KEY = ""  # @param {type:"string"}
# @markdown Used by Auto Gallery prompt planning, reference validation, and prompt repair. Blank falls back to Colab Secrets or saved private settings.
TAVILY_API_KEY = ""  # @param {type:"string"}
# @markdown Used by Auto Gallery reference search. Blank falls back to Colab Secrets or saved private settings.
GEMINI_MODEL = ""  # @param {type:"string"}
# @markdown Optional. Leave blank to auto-pick an available Gemini Flash model for your key.
# @markdown ---
# @markdown ### Avatar Gallery search controls
AVATAR_REFERENCE_DOMAINS = "instagram.com"  # @param {type:"string"}
# @markdown Comma-separated domains for Tavily. Use `instagram.com` for the fashion-reference workflow, or blank for open web.
AVATAR_REFERENCE_TIME_RANGE = "month"  # @param ["", "day", "week", "month", "year"]
# @markdown Optional Tavily recency filter.
AVATAR_SEARCH_ROUNDS = 3  # @param {type:"integer"}
# @markdown More rounds improve the chance of reaching the requested gallery count but use more API calls. Range enforced: 1-5.
AVATAR_GALLERY_RETRIES = 2  # @param {type:"integer"}
# @markdown Prompt-repair retries per gallery image after SmolVLM rejects it. Range enforced: 0-3.
AVATAR_MAX_CANDIDATE_DOWNLOADS = 60  # @param {type:"integer"}
# @markdown Max downloaded candidate references per search round. Range enforced: 10-120.
# @markdown ---
# @markdown ### Avatar image-to-text controls
AVATAR_VISION_MAX_EDGE = 768  # @param [768, 1024, 1536]
# @markdown SmolVLM inference resize. 768 is safest with FLUX on a free T4; 1024/1536 are slower and heavier.
AVATAR_VISION_MAX_TOKENS = 900  # @param {type:"integer"}
# @markdown Max SmolVLM answer length for face/body specs and validation.
PROJECT_REPO_URL = "https://github.com/itskrishnamalhotra-stack/FreeFakeStudio.git"  # @param {type:"string"}
# @markdown Use your modified fork/repo here. The original upstream should not overwrite these changes.
# @markdown ---

import os
import subprocess
from pathlib import Path

from google.colab import drive
from IPython.display import HTML, display

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

workspace = Path(WORKSPACE_DIR).expanduser().resolve()
workspace.mkdir(parents=True, exist_ok=True)
app_dir = workspace / "app"

display(HTML(f"""
<div style='font-family:Inter,system-ui,sans-serif;max-width:680px;padding:14px 16px;border:1px solid #d0d7de;border-radius:10px'>
  <b>FreeFakeStudio</b><br>
  Google Drive mounted. Workspace: <code>{workspace}</code>
</div>
"""))

def run(args, check=True):
    result = subprocess.run(args, text=True, capture_output=True)
    if check and result.returncode != 0:
        raise RuntimeError((result.stderr or result.stdout)[-1500:])
    return result

if not (app_dir / "launch.py").exists():
    if app_dir.exists() and any(app_dir.iterdir()):
        raise RuntimeError(
            f"{app_dir} exists but launch.py is missing. Move/rename it or set a new WORKSPACE_DIR."
        )
    run(["git", "clone", "--depth", "1", PROJECT_REPO_URL, str(app_dir)])
elif UPDATE_APP:
    run(["git", "-C", str(app_dir), "pull", "--ff-only"])

if not (app_dir / "launch.py").exists():
    raise RuntimeError("FreeFakeStudio app copy is incomplete: launch.py is missing.")

os.environ["FFS_WORKSPACE"] = str(workspace)
os.environ["FFS_UPDATE"] = "1" if UPDATE_APP else ""
os.environ["FFS_REPAIR"] = "1" if REPAIR_INSTALL else ""
os.environ["FFS_NGROK_AUTHTOKEN"] = NGROK_AUTH_TOKEN.strip()
os.environ["FFS_FLUX_ENCODER_MODE"] = FLUX_ENCODER.strip().lower()
os.environ["FFS_FLUX_CUSTOM_ENCODER_URL"] = FLUX_CUSTOM_ENCODER_URL.strip()
if HUGGINGFACE_TOKEN.strip():
    os.environ["HF_TOKEN"] = HUGGINGFACE_TOKEN.strip()
if GEMINI_API_KEY.strip():
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY.strip()
if TAVILY_API_KEY.strip():
    os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY.strip()
if GEMINI_MODEL.strip():
    os.environ["FFS_GEMINI_MODEL"] = GEMINI_MODEL.strip()
os.environ["FFS_AVATAR_REFERENCE_DOMAINS"] = AVATAR_REFERENCE_DOMAINS.strip()
os.environ["FFS_AVATAR_REFERENCE_TIME_RANGE"] = AVATAR_REFERENCE_TIME_RANGE.strip()
os.environ["FFS_AVATAR_SEARCH_ROUNDS"] = str(AVATAR_SEARCH_ROUNDS)
os.environ["FFS_AVATAR_GALLERY_RETRIES"] = str(AVATAR_GALLERY_RETRIES)
os.environ["FFS_AVATAR_MAX_CANDIDATE_DOWNLOADS"] = str(AVATAR_MAX_CANDIDATE_DOWNLOADS)
os.environ["FFS_AVATAR_VISION_MAX_EDGE"] = str(AVATAR_VISION_MAX_EDGE)
os.environ["FFS_AVATAR_VISION_MAX_TOKENS"] = str(AVATAR_VISION_MAX_TOKENS)

exec(compile((app_dir / "launch.py").read_text(), str(app_dir / "launch.py"), "exec"))
